In [1]:
print("Hello Conor")

Hello Conor


In [9]:
#need to import packages

try:
    import pyarrow
except ModuleNotFoundError:
    ! pip install pyarrow
    import pyarrow





import numpy as np
import pandas as pd
from urllib.request import urlopen
from pathlib import Path
import gzip


In [ ]:
#Now we read in the files, set it up to be reproducible and changing paths is easy
city = 'London'
ymd = '20260917'
host = 'https://orca.casa.ucl.ac.uk'
url = f'{host}/~jreades/data/{ymd}-{city}-listings.csv.gz'

#reproducible path read in

if Path.cwd().name == 'data_prep':
    dir_data = Path('../data')
else:
    dir_data = Path('data')
dir_raw = dir_data / 'raw'
dir_clean = dir_data / 'clean'

print(f'Data folder is: {dir_data}')
print(f'Raw data folder is: {dir_raw}')
print(f'Clean data folder is: {dir_clean}')

#since we are using multiple years each year needs a folder
#use the same file name as the url so it matches the qmd and doesn't download twice
path = dir_raw / ymd[:4] / Path(url).name
print(f'Writing out data to: {path}')


In [ ]:
#next we need to select the columns we need from our datasets

cols = ['id', 'listing_url', 'last_scraped', 'name',
    'description', 'host_id', 'host_name', 'host_since',
    'host_location', 'host_about', 'host_is_superhost',
    'host_listings_count', 'host_total_listings_count',
    'host_verifications', 'latitude', 'longitude',
    'property_type', 'room_type', 'accommodates',
    'bathrooms', 'bathrooms_text', 'bedrooms', 'beds',
    'amenities', 'price', 'minimum_nights', 'maximum_nights',
    'availability_365', 'number_of_reviews',
    'first_review', 'last_review', 'review_scores_rating',
    'license', 'reviews_per_month']

#extra columns that might be interesting for answering the qs
extra_cols = ['neighbourhood_cleansed',
    'number_of_reviews_ltm', 'calculated_host_listings_count',
    'calculated_host_listings_count_entire_homes', 'estimated_occupancy_l365d',
    'estimated_revenue_l365d']

cols = cols + extra_cols

print (len(cols))




In [15]:

#now we need to download the data and read it in.

def test_gzip(file_path):
    """Test if a file is a valid gzip file."""
    try:
        with gzip.open(file_path, 'rb') as f:
            f.read(1024 * 1024)
        return True
    except (OSError, EOFError):
        return False

def download_and_read_csvs(url, path):
    if not path.exists() or not test_gzip(path):
        if path.exists():
            path.unlink()  #remove file if it does exist but is not a valid gzip file
        print(f'Downloading data from {url}...')
        path.parent.mkdir(parents=True, exist_ok=True)  #make sure the directory exists
        with urlopen(url) as response, open(path, 'wb') as out_file:
            while chunk := response.read(1024 * 1024):
                out_file.write(chunk)

download_and_read_csvs(url, path)

df = pd.read_csv(path, compression = 'gzip', low_memory = False, usecols = cols, dtype = {'id' : str, 'host_id' : str})

print(f' dataframe is {df.shape[0]} rows by {df.shape[1]} columns')


 dataframe is 91954 rows by 40 columns


In [ ]:
#now we need to drop rows that have no id.

df = df.dropna(subset=['id'])
print(f'Our dataframe now has {df.shape[0]} rows, {df.shape[1]} columns')

#now we check the columns
print(df.isnull().sum().sort_values(ascending = False)[:12])

empty_cols = df.columns[df.isnull().all().to_list()]
print(f'These columns are empty: {empty_cols}')
#also drop host_about, it's missing for about half the listings and would push normal rows over the cutoff
df = df.drop(columns=list(empty_cols) + ['host_about'])
print(f'Our dataframe now has {df.shape[0]} rows, {df.shape[1]} columns')

#count missing values per row over the practical's columns only (not extra_cols)
probability = df.drop(columns=extra_cols).isnull().sum(axis = 1)

probability.plot.hist(bins = 100, figsize = (10, 5), title = 'Histogram of missing values per row')
print(probability.value_counts().sort_index())

#now we get rid of the rows that have more than 5 missing values from the practical.
cutoff = 5
dropped = df[probability > cutoff]
print(f'Dropping {(probability > cutoff).sum()} rows with more than {cutoff} missing values')

df = df[probability <= cutoff]
print(f'Our dataframe now has {df.shape[0]} rows, {df.shape[1]} columns')

print(f"Dropped rows with any reviews: {(dropped.number_of_reviews > 0).sum()}")


In [26]:
#we need to ensure that the types are correct for our columns.
#used chatgpt here to help debug code where I needed to add filters in first.

#boolieans
bool_cols = ['host_is_superhost']
for b in bool_cols:
    df[b] = df[b].replace({'t': True, 'f': False}).astype(bool)

#dates
dates = [d for d in ['last_scraped', 'host_since', 'first_review', 'last_review'] if d in df.columns]
for d in dates:
    df[d] = pd.to_datetime(df[d], errors='coerce')

#categories, plus the boroughs
cats = ['neighbourhood_cleansed', 'property_type', 'room_type']
for c in cats:
    df[c] = df[c].astype('category')

#money
money_cols = ['price']
for m in money_cols:
    if df[m].dtype == object:
        df[m] = df[m].str.replace('$', '', regex=False).str.replace(',', '', regex=False).astype(float)

#ints
int_cols = ['host_listings_count','accommodates','beds','minimum_nights',
         'maximum_nights','availability_365','number_of_reviews',
         'number_of_reviews_ltm','calculated_host_listings_count',
         'calculated_host_listings_count_entire_homes',
         'estimated_occupancy_l365d']
int_cols = [i for i in int_cols if i in df.columns]
for i in int_cols:
    try:
        df[i] = df[i].astype(float).astype('int')
    except ValueError as e:
        print(f"Error converting column {i} to int: {e}")
        df[i] = df[i].astype('float').astype('Int64') #just in case there are NaNs, we can use the nullable integer type

df.info()



Error converting column beds to int: Cannot convert non-finite values (NA or inf) to integer. Replace or remove non-finite values or cast to an integer type that supports these values (e.g. 'Int64')
Error converting column minimum_nights to int: Cannot convert non-finite values (NA or inf) to integer. Replace or remove non-finite values or cast to an integer type that supports these values (e.g. 'Int64')
Error converting column maximum_nights to int: Cannot convert non-finite values (NA or inf) to integer. Replace or remove non-finite values or cast to an integer type that supports these values (e.g. 'Int64')
<class 'pandas.DataFrame'>
Index: 79455 entries, 0 to 91951
Data columns (total 35 columns):
 #   Column                                       Non-Null Count  Dtype         
---  ------                                       --------------  -----         
 0   id                                           79455 non-null  str           
 1   listing_url                               

In [29]:
#I want to now flag listings that might be hotels or hostels, as they are not the focus of our analysis. We will use a combination of property type and room type to identify these listings.
df['likely_hotel'] = (
    df.property_type.str.contains('hostel|hotel|b&b|guesthouse|inn|motel|bed and breakfast|bnb', case=False, na=False) |
    df.room_type.str.contains('shared room|dormitory', case=False, na=False) |
    df.name.str.contains('hostel|hotel|b&b|guesthouse|inn|motel|bed and breakfast|bnb|student', case=False, na=False) |
    df.description.str.contains('hostel|hotel|b&b|guesthouse|inn|motel|bed and breakfast|bnb|student', case=False, na=False) |
    (df.room_type == 'Hotel room')
)

#I also want to test if they are active or not where they are reviewed at least once in the last year. I will use the estimated occupancy and revenue columns to help with this, as well as the number of reviews in the last 12 months.
df['active_ltm'] = (df['number_of_reviews_ltm'] > 0)

#now to check if they are an entirehome/apartment.
df['entire_home'] = (df['room_type'] == 'Entire home/apt')

#now to check if they are an active entire home/apartment.
df['active_entire_home'] = df['entire_home'] & df['active_ltm']

#need to flag 0 bedrooms as they are probably studios.
df['studio'] = (df['bedrooms'] == 0)

#also need to flag listings that have absurd prices as these are no longer available or block bookings.
df['price_clean'] = df['price']
df['absurd_price'] = (df['price'] > 5000)
df.loc[df['price'] > 5000, 'price_clean'] = np.nan

print(f'Likely hotels/hostels: {df["likely_hotel"].sum()}')
print(f'Studios: {df["studio"].sum()}')
print(f'Absurd prices: {df["absurd_price"].sum()}')

Likely hotels/hostels: 8773
Studios: 2807
Absurd prices: 85


In [31]:
#saving time is my favorite so here we go
csv_out = dir_clean / path.name
pq_out = dir_clean / path.name.replace('.csv.gz', '.parquet')

if not csv_out.parent.exists():
    csv_out.parent.mkdir(parents=True, exist_ok=True)

if not pq_out.parent.exists():
    pq_out.parent.mkdir(parents=True, exist_ok=True)

df.to_csv(csv_out, index=False, compression='gzip')
df.to_parquet(pq_out, index=False, engine='pyarrow')
print(f'Cleaned data saved to {csv_out} and {pq_out}')
print(f'Cleaned data has {df.shape[0]} rows and {df.shape[1]} columns')


Cleaned data saved to ..\data\clean\20260917--London--listings.csv.gz and ..\data\clean\20260917--London--listings.parquet
Cleaned data has 79455 rows and 42 columns


In [32]:
#now we need to check that everything worked.

n_all = df.shape[0]
n_reviewed = df['number_of_reviews'].notnull().sum()
n_active = df['active_ltm'].sum()
n_active_entire = df['active_entire_home'].sum()
n_homes = (df.active_entire_home & ~df.likely_hotel).sum()

print(f'Total listings: {n_all}')
print(f'Listings with reviews: {n_reviewed}')
print(f'Active listings (reviewed in last 12 months): {n_active}')
print(f'Active entire home listings: {n_active_entire}')
print(f'Active homes (not hotels): {n_homes}')

Total listings: 79455
Listings with reviews: 79455
Active listings (reviewed in last 12 months): 47114
Active entire home listings: 33402
Active homes (not hotels): 30532
